### Scrape Student Link

In [2]:
from selenium import webdriver
from selenium.webdriver.common.keys import Keys
from selenium.webdriver.common.by import By

In [3]:
import os
from dotenv import load_dotenv

load_dotenv()
BU_USERNAME=os.getenv("BU_USERNAME")
BU_PASSWORD=os.getenv("BU_PASSWORD")

In [4]:
import time
from selenium.webdriver.common.keys import Keys
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC

STUDENT_LINK = "https://student.bu.edu/MyBU"

def login_student_link(driver: webdriver.Firefox):

    wait = WebDriverWait(driver, 10)

    username_input = wait.until(EC.visibility_of_element_located((By.ID, "j_username")))
    username_input.clear()
    username_input.send_keys(BU_USERNAME)

    pass_input = wait.until(EC.visibility_of_element_located((By.ID, "j_password")))
    pass_input.clear()
    pass_input.send_keys(BU_PASSWORD)

    submit_btm = wait.until(EC.visibility_of_element_located((By.CLASS_NAME, "input-submit")))
    submit_btm.click()

    wait_duo = WebDriverWait(driver, 30)
    trust_btn = wait_duo.until(EC.visibility_of_element_located((By.ID, "trust-browser-button")))
    trust_btn.click()
    

In [5]:
class MessageRow(Exception):
    pass

In [11]:
from selenium.webdriver.support.ui import Select
from selenium.webdriver.remote.webelement import WebElement
import pandas as pd

def get_from_selector(wait, selector):
    try:
        return wait.until(EC.visibility_of_element_located((By.CSS_SELECTOR, selector)))
    except Exception as e:
        raise e

def get_to_add_course_plan_from_home(driver):
    wait = WebDriverWait(driver, 10)

    academics_link = get_from_selector(wait, "li.slds-is-relative:nth-child(2) > community_navigation-global-navigation-item:nth-child(1) > a:nth-child(1)")
    academics_link.click()

    course_schedule_link = get_from_selector(wait, "community_navigation-tile-menu-item.comm-tile-menu__item:nth-child(5) > a:nth-child(1)")
    course_schedule_link.click()

    academics = get_from_selector(wait, "body > table:nth-child(1) > tbody:nth-child(1) > tr:nth-child(2) > td:nth-child(1) > table:nth-child(1) > tbody:nth-child(1) > tr:nth-child(1) > td:nth-child(2) > a:nth-child(1)")
    academics.click()

    spring_reg = get_from_selector(wait, "body > table:nth-child(2) > tbody:nth-child(1) > tr:nth-child(2) > td:nth-child(1) > table:nth-child(1) > tbody:nth-child(1) > tr:nth-child(1) > td:nth-child(1) > font:nth-child(1) > a:nth-child(17)")
    spring_reg.click()

    reg_options = get_from_selector(wait, "body > table:nth-child(4) > tbody:nth-child(1) > tr:nth-child(2) > td:nth-child(1) > a:nth-child(2)")
    reg_options.click()

    plan = get_from_selector(wait, "body > center:nth-child(4) > table:nth-child(2) > tbody:nth-child(1) > tr:nth-child(2) > td:nth-child(2) > a:nth-child(1)")
    plan.click()

    add = get_from_selector(wait, "body > table:nth-child(8) > tbody:nth-child(1) > tr:nth-child(15) > td:nth-child(1) > a:nth-child(1)")
    add.click()


def get_cell_text_val(row: WebElement, cell_id: int):
    td = get_cell(row, cell_id)
    try:
        a_tags = td.find_elements(By.TAG_NAME, f"a")

        if a_tags:
            return a_tags[0].text
        else:
            return td.text
    except Exception as e:
        raise e

def get_title_instructor(row: WebElement):
    td = get_cell(row, 3)
    try:
        a_tags = td.find_elements(By.TAG_NAME, "a")

        if a_tags:
            html = td.get_attribute("innerHTML")
            title = html.split('<br>')[0]
            instructor = a_tags[0].text
        else:
            fields = td.text.split('\n')
            title = fields[0]
            instructor = "" if len(fields) <= 1 else fields[1]

        return title, instructor
    except Exception as e:
        raise e


def get_cell(row: WebElement, cell_id: int):
    try:
        return row.find_elements(By.TAG_NAME, "td")[cell_id]
    except Exception as e:
        raise e
    


def parse_row(row: WebElement):

    msg_cell = get_cell(row, 1)

    if int(msg_cell.get_attribute('colspan')) > 1:
        raise MessageRow

    class_code = get_cell_text_val(row, 2)
    title, instructor = get_title_instructor(row)
    open_seats = get_cell_text_val(row, 5)
    num_credits = get_cell_text_val(row, 6)
    course_type = get_cell_text_val(row, 7)
    building = get_cell_text_val(row, 8)
    room = get_cell_text_val(row, 9)
    day = get_cell_text_val(row, 10)
    start_time = get_cell_text_val(row, 11)
    stop_time = get_cell_text_val(row, 12)
    notes = get_cell_text_val(row, 13)

    return {
        "class code": class_code,
        "title": title,
        "instructor": instructor,
        "open seats": open_seats,
        "credits": num_credits,
        "course type": course_type,
        "building": building,
        "room": room,
        "day": day,
        "start time": start_time,
        "stop time": stop_time,
        "notes": notes
    }


    
def scrape(driver: webdriver.Firefox):
    wait = WebDriverWait(driver, 2)
    select = get_from_selector(wait, "body > table:nth-child(5) > tbody:nth-child(1) > tr:nth-child(2) > td:nth-child(3) > form:nth-child(2) > table:nth-child(7) > tbody:nth-child(1) > tr:nth-child(2) > td:nth-child(1) > select:nth-child(1)")
    select_obj = Select(select)
    select_obj.select_by_visible_text("CAS")

    colleges = [
        "CAS",
        "CDS",
        "CFA",
        "CGS",
        "COM",
        "ENG",
        "EOP",
        "GMS",
        "GRS",
        "HUB",
        "KHC",
        "LAW",
        "MET",
        "OTP",
        "PDP",
        "QST",
        "SAR",
        "SDM",
        "SED",
        "SHA",
        "SPH",
        "SSW",
        "STH"]

    go = get_from_selector(wait, "body > table:nth-child(5) > tbody:nth-child(1) > tr:nth-child(2) > td:nth-child(3) > form:nth-child(2) > table:nth-child(7) > tbody:nth-child(1) > tr:nth-child(2) > td:nth-child(6) > input:nth-child(1)")
    go.click()

    data = []
    college_ind = 0

    while True:
        get_from_selector(wait, "body > form:nth-child(5) > table:nth-child(1)")
        rows = driver.find_elements(By.CSS_SELECTOR, "body > form:nth-child(5) > table:nth-child(1) > tbody:nth-child(1) > tr")

        for i, row in enumerate(rows):
            # Skip Header
            if i == 0:
                continue

            try:
                row_data = parse_row(row)
                data.append(row_data)
            except MessageRow:
                pass

            if len(data) % 500 == 0:
                df = pd.DataFrame(data)
                df.to_csv(f'data/class-{len(data)}.csv')


        college_input = get_from_selector(wait, "body > form:nth-child(5) > center:nth-child(6) > table:nth-child(1) > tbody:nth-child(1) > tr:nth-child(1) > td:nth-child(3) > input:nth-child(3)")
        show_more = get_from_selector(wait, "body > form:nth-child(5) > center:nth-child(6) > table:nth-child(1) > tbody:nth-child(1) > tr:nth-child(1) > td:nth-child(2) > input:nth-child(1)")

        if college_input.get_attribute("value") != "":
            show_more.click()
        elif college_ind + 1 < len(colleges):
            college_ind += 1
            college_input.send_keys(colleges[college_ind])
        else:
            break


    return data
            

In [12]:
options = webdriver.FirefoxOptions()
driver = webdriver.Firefox(options=options)
driver.get(STUDENT_LINK)

if driver.title != "Home":
    login_student_link(driver)

get_to_add_course_plan_from_home(driver)
data = scrape(driver)
driver.quit()


In [13]:
df = pd.DataFrame(data)
df.to_csv('data/courses-spring2024.csv')